# 1. INITIALISATION DE L'ENVIRONNEMENT

In [15]:
# Importation des librairires pertinentes
import pandas as pd
import numpy as np
import glob
import os
import sqlite3
from sqlalchemy import create_engine

print("✅ Environnement technique initialisé et configuré.")

✅ Environnement technique initialisé et configuré.


# 2. CHARGEMENT DES DONNEES

In [16]:
# Configuration de la fonction de chargement des données
def load_csv(path) :
    try:
        data=pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
        if data.empty:
            return None
        data.columns=data.columns.astype(str).str.strip().str.lower()
        data=data.apply(lambda x: x.str.strip() if x.dtype=="object" else x)
        data=data.dropna(how="all").reset_index(drop=True)        
        return data
    except Exception as e :
        return None
    
# Configuration de la fonction de gestion de dossier
def load_all_csvs(folder_path) :
    files_pattern = os.path.join(folder_path, "*.csv")
    files_csv = glob.glob(files_pattern)
    df_dict = {}
    files_failed=[]

    for file_path in files_csv:
        file_name = os.path.basename(file_path).replace(".csv", "")
        df=load_csv(file_path)

        if df is not None :
            df_dict[file_name]=df
            print(f"\tChargement réussi : {file_name} ({len(df)} lignes et {len(df.columns)} colonnes)")
        else :
            files_failed.append(file_name)
            print(f"\tAttention : Le fichier {file_name} n'a pas pu être chargé (vide ou erreur).")

    return df_dict, files_failed


In [17]:
print("⏳ Chargement des données...")

folder_path="Data/Raw"

df_dict, files_failed=load_all_csvs(folder_path)

if len(files_failed)==0:
    print(f"✅ {len(df_dict)} fichiers chargés avec succès !")
else:
    print(f"✅ {len(df_dict)} fichiers chargés avec succès !")
    print(f"❌ {len(files_failed)} fichiers non chargés !")


⏳ Chargement des données...
	Chargement réussi : customers (8000 lignes et 6 colonnes)
	Chargement réussi : products (34 lignes et 7 colonnes)
	Chargement réussi : stock_movements (940531 lignes et 8 colonnes)
	Chargement réussi : tables (56 lignes et 4 colonnes)
	Chargement réussi : promotions (3 lignes et 4 colonnes)
	Chargement réussi : ingredients (63 lignes et 5 colonnes)
	Chargement réussi : services (3 lignes et 5 colonnes)
	Chargement réussi : recipes (87 lignes et 6 colonnes)
	Chargement réussi : employees (48 lignes et 9 colonnes)
	Chargement réussi : sales (1158154 lignes et 15 colonnes)
✅ 10 fichiers chargés avec succès !


# 3. Prépartion et Nettoyage des données (Data Cleaning)

In [18]:
# Audit de la structure native
print("--------------- Visualisation technique global des datasets -------------")

df_list=[]
sorted_names=sorted(df_dict,key=lambda name:df_dict[name].isna().sum().sum()/df_dict[name].size,reverse=True)
for df_name in sorted_names:
    df=df_dict[df_name]
    nb_cells=df.size
    nb_lines=df.shape[0]
    nb_rows=df.shape[1]
    nb_doublons=df.duplicated().sum()
    nb_na=df.isna().sum().sum()
    pct_na=round((df.isna().sum().sum()/df.size)*100 if nb_cells>0 else 0,2)
    df_list.append({"DF":df_name,"CELLULES":nb_cells,"LIGNES":nb_lines,"COLONNES":nb_rows,"DOUBLONS":nb_doublons,"NA":nb_na,"% NA":pct_na})
df_audit=pd.DataFrame(df_list)
df_total=pd.DataFrame([{"DF":"TOTAL","CELLULES":df_audit["CELLULES"].sum(),"LIGNES":df_audit["LIGNES"].sum(),"COLONNES":df_audit["COLONNES"].sum(),"DOUBLONS":df_audit["DOUBLONS"].sum(),"NA":df_audit["NA"].sum(),"% NA":df_audit["NA"].sum()/df_audit["CELLULES"].sum()*100}])
audit_df=pd.concat([df_audit,df_total]).reset_index(drop=True)
display((audit_df.style.format({"CELLULES":lambda x: f"{x:,.0f}".replace(",", " "),"LIGNES":lambda x: f"{x:,.0f}".replace(",", " "),"COLONNES":lambda x: f"{x:,.0f}".replace(",", " "),"DOUBLONS":lambda x: f"{x:,.0f}".replace(",", " "),"NA":lambda x: f"{x:,.0f}".replace(",", " "),"% NA":"{:.2f}%"}).hide(axis="index")).background_gradient(subset=["% NA"],cmap="YlOrRd"))

--------------- Visualisation technique global des datasets -------------


DF,CELLULES,LIGNES,COLONNES,DOUBLONS,NA,% NA
customers,48 000,8 000,6,0,0,0.00%
products,238,34,7,0,0,0.00%
stock_movements,7 524 248,940 531,8,0,0,0.00%
tables,224,56,4,0,0,0.00%
promotions,12,3,4,0,0,0.00%
ingredients,315,63,5,0,0,0.00%
services,15,3,5,0,0,0.00%
recipes,522,87,6,0,0,0.00%
employees,432,48,9,0,0,0.00%
sales,17 372 310,1 158 154,15,85 263,0,0.00%


In [19]:
# Paramètrage de la fonction pour vérfier la qualité d'un dataset
def check_data_quality(df,name="Dataset"):
    print(f"--------------------------------------------------------------- Visualisation technique de {name} ---------------------------------------------------------------")
    audit_cols=pd.DataFrame({"VARIABLE":df.columns,
                             "TYPE":[str(type) for type in df.dtypes],
                             "DOUBLON":[df[col].duplicated().sum() for col in df.columns],
                             "NA":df.isna().sum().to_numpy(),
                             "% NA":(df.isna().sum().to_numpy()/len(df)*100),
                             "MODALITE":df.nunique().to_numpy(),
                             "APERCU":[df[col].unique()[:3].tolist() for col in df.columns]}).sort_values(by="% NA",ascending=False)
    summary_data=pd.DataFrame({"VARIABLE":["--- GLOBAL ---"],
                               "TYPE":"-",
                               "DOUBLON":df.duplicated().sum(),
                               "NA":[df.isna().sum().sum()],
                               "% NA":[(df.isna().sum().sum()/df.size*100)],
                               "MODALITE":"-",
                               "APERCU":[f"{len(df):,.0f}".replace(",", " ")+" lignes et "+f"{df.shape[1]:,.0f}".replace(",", " ")+" colonnes"]
                               })
    audit_final=pd.concat([audit_cols,summary_data],ignore_index=True)
    audit_style=audit_final.style.format({"DOUBLON":lambda x: f"{x:,.0f}".replace(",", " "),"NA":lambda x: f"{x:,.0f}".replace(",", " "),"% NA":"{:.2f}%","MODALITE":lambda x: f"{x:,.0f}".replace(",", " ") if isinstance(x,(int,float)) else x}).background_gradient(subset=["% NA"],cmap="YlOrRd")
    return display(audit_style)

In [20]:
# Visualisation technique de chaque dataset
for name, df in df_dict.items():
    check_data_quality(df, name=name)

--------------------------------------------------------------- Visualisation technique de customers ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,customer_id,int64,0,0,0.00%,8 000,"[1, 2, 3]"
1,customer_code,object,0,0,0.00%,8 000,"['CUS001', 'CUS002', 'CUS003']"
2,customer_name,object,152,0,0.00%,7 848,"['Claudine Weiss-Bernier', 'Franck Devaux', 'Adrienne Torres']"
3,customer_type,object,7 997,0,0.00%,3,"['Local', 'Professionnel', 'Touriste']"
4,customer_price_sensitivity,float64,7 929,0,0.00%,71,"[0.97, 0.34, 0.88]"
5,customer_visit_frequency,int64,7 990,0,0.00%,10,"[2, 3, 1]"
6,--- GLOBAL ---,-,0,0,0.00%,-,8 000 lignes et 6 colonnes


--------------------------------------------------------------- Visualisation technique de products ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,product_id,int64,0,0,0.00%,34,"[1, 2, 3]"
1,product_code,object,0,0,0.00%,34,"['PCT001', 'PCT002', 'PCT003']"
2,product_name,object,0,0,0.00%,34,"['Planche Charcuterie', 'Burrata à la Truffe', 'Ceviche de Dorade']"
3,product_type,object,19,0,0.00%,15,"['Charcuterie', 'Fromage', 'Poisson']"
4,product_cat,object,30,0,0.00%,4,"['Entrée', 'Plat', 'Dessert']"
5,product_segment,object,30,0,0.00%,4,"['Standard', 'Premium', 'Budget']"
6,product_price,float64,9,0,0.00%,25,"[16.0, 19.5, 18.0]"
7,--- GLOBAL ---,-,0,0,0.00%,-,34 lignes et 7 colonnes


--------------------------------------------------------------- Visualisation technique de stock_movements ---------------------------------------------------------------


...

,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,recipe_id,int64,53,0,0.00%,34,"[1, 2, 3]"
1,recipe_code,object,53,0,0.00%,34,"['RCP001', 'RCP002', 'RCP003']"
2,product_id,int64,53,0,0.00%,34,"[1, 2, 3]"
3,ingredient_id,int64,24,0,0.00%,63,"[1, 2, 3]"
4,recipe_quantity,int64,67,0,0.00%,20,"[50, 125, 2]"
5,recipe_unit,object,84,0,0.00%,3,"['g', 'ml', 'u']"
6,--- GLOBAL ---,-,0,0,0.00%,-,87 lignes et 6 colonnes


--------------------------------------------------------------- Visualisation technique de employees ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,employee_id,int64,0,0,0.00%,48,"[1, 2, 3]"
1,employee_code,object,0,0,0.00%,48,"['EMP001', 'EMP002', 'EMP003']"
2,employee_name,object,0,0,0.00%,48,"['Sophie Lemonnier', 'Dominique de la Roy', 'Thierry Giraud']"
3,employee_role,object,37,0,0.00%,11,"['Chef de cuisine', 'Second de cuisine', 'Chef de partie']"
4,employee_team,int64,46,0,0.00%,2,"[1, 2]"
5,employee_department,object,46,0,0.00%,2,"['Cuisine', 'Salle']"
6,employee_fixed_monthly_salary,int64,39,0,0.00%,9,"[3500, 2600, 2100]"
7,employee_monthly_hours_contract,float64,47,0,0.00%,1,[151.67]
8,employee_hourly_rate,float64,39,0,0.00%,9,"[23.08, 17.14, 13.85]"
9,--- GLOBAL ---,-,0,0,0.00%,-,48 lignes et 9 colonnes


--------------------------------------------------------------- Visualisation technique de sales ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,order_id,int64,848 826,0,0.00%,309 328,"[1, 2, 3]"
1,order_code,object,848 826,0,0.00%,309 328,"['ORD00001', 'ORD00002', 'ORD00003']"
2,datetime,object,901 462,0,0.00%,256 692,"['2020-01-01 12:56:00', '2020-01-01 12:19:00', '2020-01-01 12:14:00']"
3,date,object,1 156 682,0,0.00%,1 472,"['2020-01-01', '2020-01-02', '2020-01-03']"
4,hour,int64,1 158 143,0,0.00%,11,"[12, 13, 14]"
5,product_id,int64,1 158 120,0,0.00%,34,"[15, 10, 1]"
6,customer_id,int64,1 150 154,0,0.00%,8 000,"[5426, 7283, 2883]"
7,table_id,int64,1 158 098,0,0.00%,56,"[11, 4, 56]"
8,service_id,int64,1 158 151,0,0.00%,3,"[1, 2, 3]"
9,promotion_id,int64,1 158 151,0,0.00%,3,"[1, 2, 3]"


### 3.1. Étude de la spécialisation et de l'affectation des équipes

In [21]:
# 1. Préparation des données
sales_and_services = pd.merge(df_dict["sales"], df_dict["services"], on="service_id", how="left")
df = pd.merge(sales_and_services, df_dict["employees"], on="employee_id", how="left")

# 2. Analyse de la charge par service/équipe (Volume Pax)
print("--- Charge de travail (Pax) par créneau ---")
display(df.groupby(["datetime", "service_id", "employee_team"])["order_pax"].sum().head(10))

# 3. Vérification de l'unicité et répartition globale
print("\n--- Unicité des équipes par service (nunique) ---")
display(df.groupby('service_id')['employee_team'].nunique())

print("\n--- Matrice d'affectation (Crosstab) ---")
display(pd.crosstab(df['service_id'], df['employee_team']))

# 4. Vérification : Les services 1 et 3 doivent avoir exactement 1 équipe unique
verif = df.groupby('service_id')['employee_team'].nunique()
regle_ok = (verif.get(1, 0) == 1) and (verif.get(3, 0) == 1)

# 5. Vérification : Est-ce qu'un rôle est toujours lié à la même équipe ?
role_team_check = df.groupby(['employee_role', 'employee_team'])['employee_id'].nunique()
print("--- Relation Rôle / Équipe ---")
display(role_team_check)

# 6. Matrice étendue : Service / Rôle / Équipe
# Cela permet de voir si le service 2 est géré par les mêmes rôles selon l'équipe
pivot_roles = pd.crosstab(
    df['service_id'], 
    [df['employee_team'], df['employee_role']]
)
print("\n--- Répartition par Service, Équipe et Rôle ---")
display(pivot_roles)

if regle_ok:
    print("✅ Tout est OK : L'affectation exclusive des services 1 et 3 est respectée.")
else:
    print("⚠️ Attention : Une anomalie d'affectation a été détectée sur les services 1 ou 3.")

--- Charge de travail (Pax) par créneau ---


datetime             service_id  employee_team
2020-01-01 12:09:00  1           1                 8
2020-01-01 12:14:00  1           1                24
2020-01-01 12:17:00  1           1                 8
2020-01-01 12:19:00  1           1                16
2020-01-01 12:45:00  1           1                 6
2020-01-01 12:48:00  1           1                 6
2020-01-01 12:53:00  1           1                32
2020-01-01 12:56:00  1           1                10
2020-01-01 13:00:00  1           1                 8
2020-01-01 13:07:00  1           1                42
Name: order_pax, dtype: int64


--- Unicité des équipes par service (nunique) ---


service_id
1    1
2    2
3    1
Name: employee_team, dtype: int64


--- Matrice d'affectation (Crosstab) ---


...

--- Relation Rôle / Équipe ---


employee_role       employee_team
Chef de rang        1                3
                    2                3
Commis de salle     1                3
                    2                3
Directeur de salle  1                1
                    2                1
Maître d'hôtel      1                1
                    2                1
Runner              1                2
                    2                2
Serveur             1                5
                    2                5
Name: employee_id, dtype: int64


--- Répartition par Service, Équipe et Rôle ---


employee_team            1                                                    \
employee_role Chef de rang Commis de salle Directeur de salle Maître d'hôtel   
service_id                                                                     
1                    99457           16951               2841           2856   
2                    27622            4621                820            799   
3                        0               0                  0              0   

employee_team                           2                                     \
employee_role Runner Serveur Chef de rang Commis de salle Directeur de salle   
service_id                                                                     
1              22265  277338            0               0                  0   
2               6106   77353        28172            4590                771   
3                  0       0       117397           19178               3151   

employee_team                                
employee_role Maître d'hôtel Runner Serveur  
service_id                                   
1                          0      0       0  
2                        814   6319   77142  
3                       3387  26478  331726

✅ Tout est OK : L'affectation exclusive des services 1 et 3 est respectée.


# 4. Exportation du modèle relationnel

Avec les 10 tables, nous pouvons distinguer 3 types de tables :
* **Tables de faits :**
  * **sales :** En-têtes des ventes (Date, Client, Montant total) avec les articles vendus
  * **stock_movements :** Flux d'inventaire (Consommation, Réapprovisionnement, Pertes)
* **Tables de dimensions :**
  * **products :** Catalogue des produits
  * **ingredients :** Liste des ingrédients
  * **customers :** Données relatives aux clients
  * **employees :** Données relatives au staff
  * **tables :** Configuration de la salle
  * **services :** Données relatives aux services déjeuner, cocktails et dîner
  * **promotions :** Offres marketing
* **Tables de liaison (Bridge):**
  * **recipes :** Tables de pont entre les products et les ingrédients

### 4.1. Vérification de la cohérence référentielle entre les tables de faits et les tables de dimensions

Cette étape assure la cohérence entre les colonnes de liaison (clés étrangères) des tables de faits et les référentiels des tables de dimensions (clés primaires). L'objectif est de garantir une intégrité référentielle totale pour éviter les erreurs de jointure dans Power BI.

In [22]:
# Définition de la fonction pour vérifier les référentiels
def check_referential_integrity(fact_df,fact_col,dim_df,dim_col):
    # Récupération des valeurs uniques
    fact_set=set(fact_df[fact_col].unique())
    dim_set=set(dim_df[dim_col].unique())
    
    # Calcul des codes orphelins (présents dans faits mais pas dans dim)
    orphans=fact_set-dim_set
    
    print(f"--- Vérification sur la clé : {fact_col} ---")
    if not orphans:
        print("✅ Succès : Tous les codes de la table de faits existent dans la dimension.")
        print(f"Nombre de codes uniques vérifiés : {len(fact_set)}")
    else:
        print(f"❌ Alerte : {len(orphans)} code(s) n'ont pas de correspondance dans la dimension !")
        print(f"Codes orphelins : {orphans}")
        
    # Optionnel : Calcul des codes inutilisés (présents dans dim mais pas dans faits)
    unused=dim_set-fact_set
    if unused:
        print(f"ℹ️ Info : {len(unused)} code(s) de la dimension ne sont pas utilisés dans les faits.")

    print("-" * 100)

Vérification de la cohérence référentielle entre la table de fait `sales` et :
* **Table de dimensions :**
  * `customers`
  * `employees`
  * `product`
  * `promotions`
  * `services`
  * `tables` 

In [23]:
# Vérification de la cohérence référentielle entre orders et les tables de dimensions mentionnées ci-dessus
check_referential_integrity(df_dict["sales"],"customer_id",df_dict["customers"],"customer_id")
check_referential_integrity(df_dict["sales"],"employee_id",df_dict["employees"],"employee_id")
check_referential_integrity(df_dict["sales"],"product_id",df_dict["products"],"product_id")
check_referential_integrity(df_dict["sales"],"promotion_id",df_dict["promotions"],"promotion_id")
check_referential_integrity(df_dict["sales"],"service_id",df_dict["services"],"service_id")
check_referential_integrity(df_dict["sales"],"table_id",df_dict["tables"],"table_id")

--- Vérification sur la clé : customer_id ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 8000
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : employee_id ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 30
ℹ️ Info : 18 code(s) de la dimension ne sont pas utilisés dans les faits.
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : product_id ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 34
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : promotion_id ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques

L'intégration entre la table de faits `sales` et les différentes tables de dimension est parfaitement alignée, garantissant ainsi l'intégrité du schéma.

Vérification de la cohérence référentielle entre la table de fait `stock_movements` et :
* **Table de dimensions :**
  * `ingredients`

In [24]:
# Vérification de la cohérence référentielle entre stock_movements et ingredients
check_referential_integrity(df_dict["stock_movements"],"ingredient_id",df_dict["ingredients"],"ingredient_id")

--- Vérification sur la clé : ingredient_id ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 63
----------------------------------------------------------------------------------------------------


L'intégration entre la table de fait `stock_movements` et la table de dimension `ingredients` est parfaitement alignée, garantissant ainsi l'intégrité du schéma.

Vérification de la cohérence référentielle entre la table birdge `recipes` et :
* **Table de dimensions :**
  * `ingredients`
  * `products`

In [25]:
# Vérification de la cohérence référentielle entre recipes et ingredients et products
check_referential_integrity(df_dict["recipes"],"ingredient_id",df_dict["ingredients"],"ingredient_id")
check_referential_integrity(df_dict["recipes"],"product_id",df_dict["products"],"product_id")

--- Vérification sur la clé : ingredient_id ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 63
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : product_id ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 34
----------------------------------------------------------------------------------------------------


L'intégration entre la table bridge `recipes` et les tables de dimension `products` et `ingredients` est parfaitement alignée, garantissant ainsi l'intégrité du schéma.

Toutes les tables sont parfaitement connectées entre elles, garantissant une cohérence totale des données. Les fichiers CSV peuvent être généré et importé dans Power BI.

### 4.2. Exportation du modèle relationnel vers des fichiers CSV

In [26]:
# Configuration de la fonction d'export des données
def export_data(dataframes):
    output_path = "Data/Processed"
    os.makedirs(output_path, exist_ok=True)
    
    # 1. Définition de la liste des tables attendues (le "Contrat")
    expected_tables = {
        "fact_": ["sales", "stock_movements"],
        "dim_": ["products", "ingredients", "customers", "employees", "services", "tables", "promotions"],
        "bridge_": ["recipes"]
    }
    
    # On aplatit la liste pour faciliter la comparaison
    all_expected = [t for sublist in expected_tables.values() for t in sublist]
    
    # 2. Contrôle de présence : Quelles tables manquent ?
    missing = [t for t in all_expected if t not in dataframes]
    if missing:
        print(f"❌ ERREUR : Tables manquantes dans le dictionnaire : {missing}")
        return # On arrête tout si le dataset est incomplet
    
    # 3. Export
    print("🚀 Validation réussie, début de l'export...\n")
    
    for name, df in dataframes.items():
        # Déterminer le préfixe
        prefix = next((p for p, tables in expected_tables.items() if name in tables), "")
        file_name = f"{prefix}{name}"
        
        df.to_csv(f"{output_path}/{file_name}.csv", index=False)
        print(f"\t📄 Exporté : {file_name}.csv")
    
    print(f"\n✅ Dataset complet exporté dans : {output_path}")

# Export des données sous csv
export_data(df_dict)

🚀 Validation réussie, début de l'export...

	📄 Exporté : dim_customers.csv
	📄 Exporté : dim_products.csv
	📄 Exporté : fact_stock_movements.csv
	📄 Exporté : dim_tables.csv
	📄 Exporté : dim_promotions.csv
	📄 Exporté : dim_ingredients.csv
	📄 Exporté : dim_services.csv
	📄 Exporté : bridge_recipes.csv
	📄 Exporté : dim_employees.csv
	📄 Exporté : fact_sales.csv

✅ Dataset complet exporté dans : Data/Processed


### 4.3. Exportation du modèle relationnel vers une base de donnée PostgreSQL

In [27]:
# Initialisation du dossier de stockage
user = "juliengrapin"
host = "localhost"
port = 5432
database = "restaurant"

engine = create_engine(
    f"postgresql+psycopg2://{user}@{host}:{port}/{database}"
)

# 1. Définition du contrat
expected_tables = {
        "fact_": ["sales", "stock_movements"],
        "dim_": ["products", "ingredients", "customers", "employees", "services", "tables", "promotions"],
        "bridge_": ["recipes"]
    }

all_expected = [t for sublist in expected_tables.values() for t in sublist]

# 2. Contrôle de présence : on vérifie si les clés sont dans df_dict
missing = [t for t in all_expected if t not in df_dict]
if missing:
    print(f"❌ ERREUR : Tables manquantes dans df_dict : {missing}")
else:
    # 3. Export
    print("⏳ Début de l'export vers la base PostgreSQL...\n")
    
    for name, df in df_dict.items():
        # Trouver le préfixe
        prefix = ""
        for p, tables in expected_tables.items():
            if name in tables:
                prefix = p
                break
        
        table_name_in_db = f"{prefix}{name}"
        
        # Export avec le nom préfixé
        df.to_sql(table_name_in_db, engine, if_exists="replace", index=False)
        print(f"\t📄 Table {table_name_in_db} exportée.")
    
    print(f"\n✅ Toutes les tables ont été injectées dans PostgreSQL")


⏳ Début de l'export vers la base PostgreSQL...

	📄 Table dim_customers exportée.
	📄 Table dim_products exportée.
	📄 Table fact_stock_movements exportée.
	📄 Table dim_tables exportée.
	📄 Table dim_promotions exportée.
	📄 Table dim_ingredients exportée.
	📄 Table dim_services exportée.
	📄 Table bridge_recipes exportée.
	📄 Table dim_employees exportée.
	📄 Table fact_sales exportée.

✅ Toutes les tables ont été injectées dans PostgreSQL


### 4.4 Exportation du modèle relationnel vers une base de donnée SQLite

In [28]:
# Initialisation du dossier de stockage
folder="Data/Processed"
file_name="restaurant_data.db"
os.makedirs(folder, exist_ok=True)
sql_path = os.path.join(folder, file_name)
conn=sqlite3.connect(sql_path)

# 1. Définition du contrat
expected_tables = {
        "fact_": ["sales", "stock_movements"],
        "dim_": ["products", "ingredients", "customers", "employees", "services", "tables", "promotions"],
        "bridge_": ["recipes"]
    }

all_expected = [t for sublist in expected_tables.values() for t in sublist]

# 2. Contrôle de présence : on vérifie si les clés sont dans df_dict
missing = [t for t in all_expected if t not in df_dict]
if missing:
    print(f"❌ ERREUR : Tables manquantes dans df_dict : {missing}")
else:
    # 3. Export
    print("⏳ Début de l'export vers la base PostgreSQL...\n")
    conn = sqlite3.connect(sql_path)
    
    for name, df in df_dict.items():
        # Trouver le préfixe
        prefix = ""
        for p, tables in expected_tables.items():
            if name in tables:
                prefix = p
                break
        
        table_name_in_db = f"{prefix}{name}"
        
        # Export avec le nom préfixé
        df.to_sql(table_name_in_db, conn, if_exists="replace", index=False)
        print(f"\t📄 Table {table_name_in_db} exportée.")
    
    conn.close()
    print(f"\n✅ Toutes les tables ont été injectées dans SQLite {file_name}")


⏳ Début de l'export vers la base PostgreSQL...

	📄 Table dim_customers exportée.
	📄 Table dim_products exportée.
	📄 Table fact_stock_movements exportée.
	📄 Table dim_tables exportée.
	📄 Table dim_promotions exportée.
	📄 Table dim_ingredients exportée.
	📄 Table dim_services exportée.
	📄 Table bridge_recipes exportée.
	📄 Table dim_employees exportée.
	📄 Table fact_sales exportée.

✅ Toutes les tables ont été injectées dans SQLite restaurant_data.db
